In [3]:
import random 
from collections import deque 
from pathlib import Path 

import gymnasium as gym
import torch 
import torchvision 
import matplotlib.pyplot as plt 
import seaborn as sns 
import torch.nn as nn 
print("Imports Complete!")

Imports Complete!


# Setup Enviroment

The first step in any RL implementation is setting up the paper. This notebook will focus on implementing the enviroment for one task which will be **cartpole**.

**Problem Statement**<br> 
 A pole is attached by an un-actuated joint to a cart, which moves along a frictionless track. The pendulum is placed upright on the cart and the goal is to balance the pole by applying forces in the left and right direction on the cart.

In [4]:
# Why use v1 and not v0. The difference between v0 and v1 is that v0 terminates at 200 steps where v1 terminated at 500
env = gym.make(
    "CartPole-v1", 
    render_mode="rgb_array"
)
env.reset(seed=123, options={"low": -0.1, "high":0.1})

(array([ 0.03647037, -0.0892358 , -0.05592803, -0.06312564], dtype=float32),
 {})

In [7]:
%%writefile ../src/dqn/model.py

import numpy as np
from collections import defaultdict

class QNetwork(nn.Module):
    def __init__(self, 
                 env: gym.Env,
                 learning_rate: float, 
                 initial_epsilon: float, 
                 epsilon_decay: float, 
                 final_epsilon: float, 
                 discount_factor: float):
        super().__init__()

        self.env = env 
        self.learning_rate = learning_rate
        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon
        self.discount_factor = discount_factor
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        # Track learning rate error 
        self.training_error = []

    def get_action(self, obs: tuple[int, int, bool]):
        """
        Choose an action through epsilon-greedy strategy

        Actions:
            Stand = 0
            Hit = 1
        """

        if np.random.random() < self.epsilon:
            return self.env.action_space.sample()

        else:
            return int(np.argmax(self.q_values[obs]))

    def update(
            self, 
            obs: tuple[int, int, bool],
            action: int, 
            reward: float, 
            terminated: bool, 
            next_obs: tuple[int, int, bool]
    ):
        """
        Update Q Values based on experience 
        """

        future_q_values = (not terminated) * np.max(self.q_values[next_obs])
        target = reward + self.discount_factor * future_q_values
        temporal_difference = target - self.q_values[obs][action]
        self.q_values[obs][action] = (
            self.q_values[obs][action] + self.learning_rate * temporal_difference
        )

        # Track Learning Progress 
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        """Reduce exploration rate after each episode"""
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

Writing ../src/dqn/model.py


In [8]:
%%writefile ../src/dqn/train.py
import gymnasium as gym 
from src.dqn.model import QNetwork

def train_agent(
        env: gym.Env,
        episodes: int,
        agent: QNetwork
):
    # Start a new hand 
    obs, info = env.reset()
    done = False 

    while not done:
        action = agent.get_action(obs)
        next_obs, reward, terminated, truncated, info = env.step(action=action)
        agent.update(obs, action, reward, terminated, next_obs)
        done = terminated or truncated

    agent.decay_epsilon()

Writing ../src/dqn/train.py
